In [4]:
import pandas as pd
import json

total_rows = sum(1 for _ in open("SGJobData.csv"))

df = pd.read_csv(
    "SGJobData.csv",
    skiprows=range(1, total_rows - 20000)
)

# --------------------------------------------------------
# Build category lookup table
# --------------------------------------------------------

lookup_rows = []

for _, row in df.iterrows():

    if pd.isna(row["categories"]):
        continue

    try:

        categories = json.loads(row["categories"])

        for c in categories:

            lookup_rows.append(
                {
                    "metadata_jobPostId": row["metadata_jobPostId"],
                    "category_id": c["id"],
                    "category_name": c["category"]
                }
            )

    except Exception:
        continue

job_categories = pd.DataFrame(lookup_rows)

job_categories.to_csv(
    "job_categories.csv",
    index=False
)

print(job_categories.head())
print(job_categories.shape)
print(df.shape)

  metadata_jobPostId  category_id                     category_name
0   MCF-2024-0741472           20                   Human Resources
1   MCF-2024-0741249            5               Banking and Finance
2   MCF-2024-0741249           19                       Hospitality
3   MCF-2024-0741469            1  Accounting / Auditing / Taxation
4   MCF-2024-0741477            1  Accounting / Auditing / Taxation
(34526, 3)
(19721, 22)


In [5]:
import pandas as pd
import json

# ----------------------------------------------------------
# 1. Load the dataset
# ----------------------------------------------------------

# Read the CSV file
#df = pd.read_csv("SGJobData.csv")

print("Original shape:", df.shape)


# ----------------------------------------------------------
# 2. Remove completely empty columns
# ----------------------------------------------------------

# Drops columns where every value is NaN
df = df.dropna(axis=1, how="all")

print("After removing empty columns:", df.shape)


# ----------------------------------------------------------
# 3. Convert date columns
# ----------------------------------------------------------

date_columns = [
    "metadata_expiryDate",
    "metadata_newPostingDate",
    "metadata_originalPostingDate",
]

# Convert text dates into datetime format
for col in date_columns:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")


# ----------------------------------------------------------
# 4. Clean text columns
# ----------------------------------------------------------

# Find all text columns
text_cols = df.select_dtypes(include="object").columns

# Remove leading/trailing spaces
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip())


# ----------------------------------------------------------
# 5. Extract category information
# ----------------------------------------------------------

def extract_categories(category_text):
    """
    Converts the JSON stored in 'categories'
    into a comma-separated string.

    Example:

    '[{"id":1,"category":"IT"},
      {"id":2,"category":"Data"}]'

    becomes

    IT, Data
    """

    # Handle missing values
    if pd.isna(category_text):
        return None

    try:
        category_list = json.loads(category_text)

        return ", ".join(
            item["category"] for item in category_list
        )

    except Exception:
        return None


def extract_category_ids(category_text):

    if pd.isna(category_text):
        return None

    try:
        category_list = json.loads(category_text)

        return ", ".join(
            str(item["id"]) for item in category_list
        )

    except Exception:
        return None


# Create two new columns
df["category_names"] = df["categories"].apply(extract_categories)
df["category_ids"] = df["categories"].apply(extract_category_ids)

print("After extracting categories:", df.shape)


# ----------------------------------------------------------
# 6. Standardise text
# ----------------------------------------------------------

columns_to_title = [
    "category_names",
    "employmentTypes",
    "positionLevels",
    "postedCompany_name",
    "status_jobStatus",
    "title",
]

for col in columns_to_title:
    df[col] = df[col].str.title()


# ----------------------------------------------------------
# 7. Remove impossible salary values
# ----------------------------------------------------------

# Salary cannot be negative
df = df[df["salary_minimum"] >= 0]

# Maximum salary must be greater than minimum salary
df = df[df["salary_maximum"] >= df["salary_minimum"]]

print("After salary validation:", df.shape)


# ----------------------------------------------------------
# 8. Remove salary outliers using IQR
# ----------------------------------------------------------

Q1 = df["average_salary"].quantile(0.25)
Q3 = df["average_salary"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df = df[
    (df["average_salary"] >= lower) &
    (df["average_salary"] <= upper)
]

print("After removing salary outliers:", df.shape)


# ----------------------------------------------------------
# 9. Remove unrealistic experience values
# ----------------------------------------------------------

df = df[
    (df["minimumYearsExperience"] >= 0) &
    (df["minimumYearsExperience"] <= 30)
]

print("After experience cleaning:", df.shape)


# ----------------------------------------------------------
# 10. Save cleaned dataset
# ----------------------------------------------------------

df.to_csv("cleaned_jobs.csv", index=False)

print("Final shape:", df.shape)
print("Cleaning completed successfully.")

Original shape: (19721, 22)
After removing empty columns: (19721, 21)
After extracting categories: (19721, 23)


After salary validation: (19721, 23)
After removing salary outliers: (18315, 23)
After experience cleaning: (18315, 23)
Final shape: (18315, 23)
Cleaning completed successfully.


In [1]:
%%writefile test_app.py
from pathlib import Path

import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt


st.set_page_config(page_title="Singapore Job Market Dashboard123", layout="wide")
@st.cache_data
def load_data():
    DATA_PATH = Path(__file__).resolve().parent
    df = pd.read_csv(
    DATA_PATH/"cleaned_jobs.csv",
    parse_dates=[
        "metadata_newPostingDate",
        "metadata_originalPostingDate",
        "metadata_expiryDate",
    ],
    )
    # Load category lookup table
    job_categories = pd.read_csv(
        DATA_PATH / "job_categories.csv"
    )
    return df, job_categories

df, job_categories = load_data()

st.title("Singapore Job Market Dashboard")
filtered_df = df.copy()
with st.sidebar:
    st.header("Filters")
    category_options = sorted(job_categories["category_name"].dropna().unique())
    selected_categories = st.multiselect(
        "Category", category_options, default=category_options
    )

    employment_options = sorted(filtered_df["employmentTypes"].dropna().unique())
    selected_employment = st.selectbox("Employment Type", ["All"] + employment_options)

    open_only = st.checkbox("Show only open postings")

    min_exp = int(filtered_df["minimumYearsExperience"].min())
    max_exp = int(filtered_df["minimumYearsExperience"].max())
    experience_range = st.slider(
        "Minimum Years Experience", min_exp, max_exp, (min_exp, max_exp)
    )

    salary_range = st.sidebar.slider(
    "Salary Range",
    int(filtered_df["salary_minimum"].min()),
    int(filtered_df["salary_maximum"].max()),
    (
        int(filtered_df["salary_minimum"].min()),
        int(filtered_df["salary_maximum"].max())
    )
    )

    filtered_df = filtered_df[
    (filtered_df["salary_maximum"] >= salary_range[0]) &
    (filtered_df["salary_minimum"] <= salary_range[1])
    ]

st.metric(
    "Job Postings",
    f"{filtered_df['metadata_jobPostId'].nunique():,}"
)

filtered_df = filtered_df[filtered_df["category_names"].isin(selected_categories)]
if selected_employment != "All":
    filtered_df = filtered_df[filtered_df["employmentTypes"] == selected_employment]
if open_only:
    filtered_df = filtered_df[filtered_df["status_jobStatus"] == "Open"]
filtered_df = filtered_df[filtered_df["minimumYearsExperience"].between(*experience_range)]

st.header("Overview")

col1, col2, col3 = st.columns(3)
total_jobs = filtered_df['metadata_jobPostId'].nunique()
col1.metric("Total Job Postings", f"{total_jobs:,}")
print(filtered_df.info())
print(filtered_df.metadata_jobPostId.value_counts())

col2.metric(
    "Average Salary",
    f"${filtered_df['average_salary'].mean():,.0f}" if len(df) else "N/A",
)
col3.metric("Total Applications", int(df["metadata_totalNumberJobApplication"].sum()))

with st.expander("View raw data"):
    st.dataframe(filtered_df)

st.header("Trends & Breakdown")

postings_by_month = (
    filtered_df.groupby(df["metadata_newPostingDate"].dt.to_period("M")).size().rename("postings")
)
postings_by_month.index = postings_by_month.index.to_timestamp()

avg_salary_by_category = (
    filtered_df.groupby("category_names")["average_salary"].mean().sort_values(ascending=False)
)

tab1, tab2, tab3 = st.tabs(["Trends", "Categories", "Salary"])

with tab1:
    st.subheader("Postings Over Time")
    st.line_chart(postings_by_month)
    st.subheader("Cumulative Postings")
    st.area_chart(postings_by_month.cumsum())

with tab2:
    st.subheader("Average Salary by Category")
    st.bar_chart(avg_salary_by_category)

with tab3:
    st.subheader("Experience vs Salary")
    st.scatter_chart(filtered_df, x="minimumYearsExperience", y="average_salary")

top_companies = filtered_df['positionLevels'].value_counts().head(5)
    # Extract the sizes (the counts) and labels (the company names) dynamically
sizes = top_companies.values
labels = top_companies.index

fig, ax = plt.subplots()
ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
ax.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
    #st.subheader("Job Type Distribution")
    # 3. Display the chart in Streamlit
st.pyplot(fig) 

Overwriting test_app.py
